# RECON Jupyter Visualizer

This notebook renders either raw RECON `.hdf5` files or processed `jpg + traj_data.pkl` trajectories without requiring a GUI window.

## Usage

1. Set `DATA_FOLDERS` to one or more folders that contain either raw `.hdf5` files or processed trajectory folders.
2. Run all cells.
3. Use the sliders or play button to move across files and timesteps.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython.display import clear_output, display

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError("Install ipywidgets in this notebook environment to use the interactive controls.") from exc


def _find_recon_datavis_src() -> Path:
    """Resolve package src regardless of Jupyter CWD (repo root vs recon_datavis dir)."""
    marker = Path("recon_datavis") / "jupyter_visualizer.py"
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        # Notebook opened under datasets/recon_datavis: ./src/recon_datavis/...
        cand = d / "src" / marker
        if cand.is_file():
            return d / "src"
        # CWD is repo root (or ancestor): .../datasets/recon_datavis/src/recon_datavis/...
        cand = d / "datasets" / "recon_datavis" / "src" / marker
        if cand.is_file():
            return d / "datasets" / "recon_datavis" / "src"
    raise FileNotFoundError(
        "Could not locate recon_datavis/src (expected recon_datavis/jupyter_visualizer.py). "
        "Open the notebook from the nwm repo or from datasets/recon_datavis."
    )


src_path = _find_recon_datavis_src()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Repo root for DATA_FOLDERS: .../nwm when src is .../datasets/recon_datavis/src
p = src_path.resolve()
repo_root = p.parent.parent.parent if p.name == "src" and p.parent.name == "recon_datavis" else Path.cwd().resolve()

from recon_datavis.jupyter_visualizer import (
    JupyterHDF5Visualizer,
    JupyterProcessedTrajectoryVisualizer,
    collect_hdf5_files,
    collect_processed_trajectory_dirs,
)


In [ ]:
DATA_FOLDERS = [
    repo_root / "datasets" / "recon_raw" / "recon_release",
    # repo_root / "datasets" / "recon_1fps_train",
]

hdf5_files = collect_hdf5_files([str(folder) for folder in DATA_FOLDERS])
trajectory_dirs = collect_processed_trajectory_dirs([str(folder) for folder in DATA_FOLDERS])

if hdf5_files:
    data_mode = "hdf5"
    visualizer = JupyterHDF5Visualizer(hdf5_files)
    frame_counts = visualizer.get_frame_count_map()
    source_items = list(visualizer.hdf5_fnames)

    if visualizer.invalid_hdf5_fnames:
        print(f"Skipped {len(visualizer.invalid_hdf5_fnames)} unreadable files")
        print(visualizer.invalid_hdf5_fnames[0])

    print(f"Loaded {len(source_items)} readable HDF5 files")
elif trajectory_dirs:
    data_mode = "processed"
    visualizer = JupyterProcessedTrajectoryVisualizer(trajectory_dirs)
    frame_counts = visualizer.get_frame_count_map()
    source_items = list(visualizer.trajectory_dirs)
    print(f"Loaded {len(source_items)} processed trajectories")
else:
    raise ValueError(f"No supported RECON sources found in: {DATA_FOLDERS}")

print(f"Data mode: {data_mode}")
print(source_items[0])
print(f"Frames in first file: {frame_counts[0]}")


In [ ]:
file_slider = widgets.IntSlider(value=0, min=0, max=len(source_items) - 1, step=1, description="File")
frame_slider = widgets.IntSlider(value=0, min=0, max=frame_counts[0] - 1, step=1, description="Frame")
play = widgets.Play(value=0, min=0, max=frame_counts[0] - 1, interval=200, description="Play")
widgets.jslink((play, "value"), (frame_slider, "value"))
output = widgets.Output()

def _sync_frame_range(*_):
    max_frame = frame_counts[file_slider.value] - 1
    frame_slider.max = max_frame
    play.max = max_frame
    if frame_slider.value > max_frame:
        frame_slider.value = max_frame

def _render(*_):
    with output:
        clear_output(wait=True)
        fig, _ = visualizer.render(file_idx=file_slider.value, timestep=frame_slider.value)
        display(fig)
        plt.close(fig)

file_slider.observe(_sync_frame_range, names="value")
file_slider.observe(_render, names="value")
frame_slider.observe(_render, names="value")

_sync_frame_range()
_render()

display(widgets.VBox([
    widgets.HBox([file_slider]),
    widgets.HBox([play, frame_slider]),
    output,
]))
